# Notebook 19: Testing the Simplicity Bias Hypothesis

## Goal
Find **"alpha"** - identify tasks/setups where learned activations (splines) excel over ReLU + SH.

## Motivation
Based on **Teney et al. (2024)** "Do We Always Need the Simplicity Bias?" (CVPR):

### Where Learned Activations Excel:
1. **Regression tasks** (not classification)
2. **High-frequency tasks** (sharp transitions, peaks)
3. **Complex functions** (higher Total Variation)
4. **Fine spatial resolution** (more detail)

### Where ReLU is Near-Optimal:
1. **Classification tasks**
2. **Smooth/low-frequency tasks** (simplicity bias matches task)

### Our Results (NB18):
- **Population density** (smooth): ReLU wins by 0.63%
- **Conclusion**: Need to test on high-frequency tasks

## Experiments
1. **Regression vs Classification** ⭐⭐⭐ (CRITICAL)
2. **High-Frequency Geographic Tasks** ⭐⭐⭐ (CRITICAL)
3. **Multi-Resolution Analysis** ⭐⭐ (HIGH PRIORITY)
4. **Function Complexity Measurement** ⭐⭐ (HIGH PRIORITY)
5. **Task Difficulty Scaling** ⭐ (MEDIUM PRIORITY)

## Expected Runtime
~5-6 hours on Colab T4 GPU

In [ ]:
# Setup
import os
import sys

if 'COLAB_GPU' in os.environ:
    !rm -rf sample_data .config satclip gpw_data etopo_60s.nc coastline_data 2>/dev/null
    !git clone https://github.com/1hamzaiqbal/satclip.git
    !pip install lightning torchgeo huggingface_hub rasterio xarray netCDF4 geopandas --quiet
    sys.path.append('./satclip/satclip')
else:
    sys.path.append(os.path.join(os.path.dirname(os.getcwd()), 'satclip'))

---
## Data Acquisition

Download all required datasets:
1. **GPW Population** (from Drive) - smooth baseline
2. **ETOPO Elevation** (~60 MB) - high-frequency task
3. **Natural Earth Coastlines** (~3 MB) - sharp transitions
4. **Temperature data** (optional) - medium frequency

**Total download**: ~63 MB, ~3-5 minutes

In [ ]:
from google.colab import drive
import zipfile
import requests
import io

print("="*70)
print("STEP 1: Mounting Google Drive")
print("="*70)
drive.mount('/content/drive')
print("✅ Drive mounted successfully\n")

In [ ]:
print("="*70)
print("STEP 2: Installing required packages")
print("="*70)
!pip install -q xarray netCDF4 rasterio geopandas
print("✅ Packages installed\n")

In [ ]:
print("="*70)
print("STEP 3: Downloading ETOPO 2022 60s Elevation Data")
print("="*70)
print("URL: https://www.ngdc.noaa.gov/thredds/fileServer/global/ETOPO2022/60s/...")
print("Size: ~60 MB")
print("Resolution: 60 arc-seconds (~2 km at equator)")
print()

!wget -q --show-progress \
    "https://www.ngdc.noaa.gov/thredds/fileServer/global/ETOPO2022/60s/60s_surface_elev_netcdf/ETOPO_2022_v1_60s_N90W180_surface.nc" \
    -O etopo_60s.nc

# Verify download
if os.path.exists('etopo_60s.nc') and os.path.getsize('etopo_60s.nc') > 1000000:
    print(f"✅ Elevation data downloaded: {os.path.getsize('etopo_60s.nc') / 1e6:.1f} MB\n")
else:
    print("❌ Download failed! Check connection or use fallback API.\n")

In [ ]:
print("="*70)
print("STEP 4: Downloading Natural Earth Coastlines")
print("="*70)
print("URL: https://naciscdn.org/naturalearth/10m/physical/...")
print("Size: ~3 MB")
print()

url = "https://naciscdn.org/naturalearth/10m/physical/ne_10m_coastline.zip"
response = requests.get(url)

if response.status_code == 200:
    with zipfile.ZipFile(io.BytesIO(response.content)) as z:
        z.extractall('coastline_data')
    print("✅ Coastline data extracted to ./coastline_data/\n")
else:
    print(f"❌ Download failed with status code: {response.status_code}\n")

In [ ]:
print("="*70)
print("STEP 5: Extracting GPW Population Density Data")
print("="*70)
print("Source: dataverse_files.zip in Google Drive")
print()

GPW_DIR = './gpw_data'
os.makedirs(GPW_DIR, exist_ok=True)

zip_path = '/content/drive/MyDrive/grad/learned_activations/dataverse_files.zip'

if os.path.exists(zip_path):
    with zipfile.ZipFile(zip_path, 'r') as z:
        # Extract only the population density file
        gpw_file = 'gpw_v4_population_density_rev11_2020_15_min.tif'
        matching = [f for f in z.namelist() if gpw_file in f]
        
        if matching:
            z.extract(matching[0], GPW_DIR)
            print(f"✅ Population data extracted: {matching[0]}\n")
        else:
            print("⚠️  GPW file not found in zip, extracting all...")
            z.extractall(GPW_DIR)
            print("✅ All GPW data extracted\n")
else:
    print(f"❌ Drive path not found: {zip_path}")
    print("Please check that dataverse_files.zip is in the correct location\n")

In [ ]:
print("="*70)
print("STEP 6 (Optional): Downloading Temperature Data")
print("="*70)
print("URL: https://springernature.figshare.com/ndownloader/files/12609182")
print("Size: ~1 MB")
print()

!wget -q --show-progress \
    "https://springernature.figshare.com/ndownloader/files/12609182" \
    -O temperature.csv

if os.path.exists('temperature.csv'):
    print("✅ Temperature data downloaded\n")
else:
    print("⚠️  Temperature download failed (optional, can skip)\n")

In [ ]:
print("="*70)
print("STEP 7: Loading and Verifying Data")
print("="*70)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image
import xarray as xr
import rasterio
import geopandas as gpd
import glob
import time
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import torch.optim as optim
from sklearn.metrics import r2_score
import positional_encoding as PE

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}\n")

success_count = 0
total_datasets = 4

# 1. Load elevation
try:
    ds_elev = xr.open_dataset('etopo_60s.nc')
    elevation_data = ds_elev['z'].values
    lats_elev = ds_elev['lat'].values
    lons_elev = ds_elev['lon'].values
    print(f"✅ Elevation: {elevation_data.shape} ({len(lats_elev)} lats × {len(lons_elev)} lons)")
    success_count += 1
except Exception as e:
    print(f"❌ Elevation loading failed: {e}")
    elevation_data = None

# 2. Load coastlines
try:
    coastlines = gpd.read_file('coastline_data/ne_10m_coastline.shp')
    print(f"✅ Coastlines: {len(coastlines)} features")
    success_count += 1
except Exception as e:
    print(f"❌ Coastlines loading failed: {e}")
    coastlines = None

# 3. Load population
try:
    Image.MAX_IMAGE_PIXELS = None
    gpw_files = glob.glob('gpw_data/**/*.tif', recursive=True)
    
    if gpw_files:
        img = Image.open(gpw_files[0])
        pop_data = np.array(img)
        h, w = pop_data.shape
        lons_pop = np.linspace(-180 + 180/w, 180 - 180/w, w)
        lats_pop = np.linspace(90 - 90/h, -90 + 90/h, h)
        print(f"✅ Population: {pop_data.shape}")
        success_count += 1
    else:
        print("❌ Population file not found")
        pop_data = None
except Exception as e:
    print(f"❌ Population loading failed: {e}")
    pop_data = None

# 4. Load temperature (optional)
try:
    temp_df = pd.read_csv('temperature.csv')
    print(f"✅ Temperature: {len(temp_df)} observations (optional)")
    success_count += 1
except Exception as e:
    print(f"⚠️  Temperature loading failed (optional): {e}")
    temp_df = None

print("\n" + "="*70)
print("DATA ACQUISITION SUMMARY")
print("="*70)
print(f"✅ Successfully loaded: {success_count}/{total_datasets} datasets")

if success_count >= 3:
    print("\n🎉 ALL CRITICAL DATA READY!")
    print("You can now proceed with experiments.")
else:
    print("\n⚠️  WARNING: Some critical data failed to load")
    print("Please check error messages above and retry failed downloads")

print("="*70)

---
## Model Definitions

Define activation functions and encoder architectures.

In [ ]:
# =============================================================================
# SPLINE ACTIVATION
# =============================================================================

class SplineActivation(nn.Module):
    """Piecewise linear spline activation with learnable knot values."""
    def __init__(self, n_knots=15, input_range=(-3.0, 3.0), init='relu'):
        super().__init__()
        self.n_knots = n_knots
        self.input_range = input_range
        
        # Fixed knot positions
        knot_x = torch.linspace(input_range[0], input_range[1], n_knots)
        self.register_buffer('knot_x', knot_x)
        
        # Learnable knot values
        if init == 'relu':
            knot_y = torch.relu(knot_x)
        elif init == 'linear':
            knot_y = knot_x.clone()
        elif init == 'zero':
            knot_y = torch.zeros(n_knots)
        else:
            knot_y = torch.randn(n_knots) * 0.1
        
        self.knot_y = nn.Parameter(knot_y)
    
    def forward(self, x):
        # Clamp input to range
        x_clamped = torch.clamp(x, self.input_range[0], self.input_range[1])
        
        # Normalize to [0, 1]
        x_norm = (x_clamped - self.knot_x[0]) / (self.knot_x[-1] - self.knot_x[0])
        x_idx = x_norm * (self.n_knots - 1)
        
        # Linear interpolation
        idx_low = torch.floor(x_idx).long()
        idx_high = torch.clamp(idx_low + 1, max=self.n_knots - 1)
        idx_low = torch.clamp(idx_low, max=self.n_knots - 1)
        
        weight = x_idx - idx_low.float()
        y_low = self.knot_y[idx_low]
        y_high = self.knot_y[idx_high]
        
        return y_low + weight * (y_high - y_low)


# =============================================================================
# SIREN ACTIVATION
# =============================================================================

class SirenLayer(nn.Module):
    """SIREN layer with sinusoidal activations."""
    def __init__(self, dim_in, dim_out, w0=1.0, is_first=False):
        super().__init__()
        self.dim_in = dim_in
        self.w0 = w0
        self.is_first = is_first
        
        self.linear = nn.Linear(dim_in, dim_out)
        self._init_weights()
    
    def _init_weights(self):
        if self.is_first:
            bound = 1.0 / self.dim_in
        else:
            bound = np.sqrt(6.0 / self.dim_in) / self.w0
        
        self.linear.weight.data.uniform_(-bound, bound)
        if self.linear.bias is not None:
            self.linear.bias.data.uniform_(-bound, bound)
    
    def forward(self, x):
        return torch.sin(self.w0 * self.linear(x))


print("✅ Activation classes loaded!")

In [ ]:
# =============================================================================
# UNIVERSAL ENCODER
# =============================================================================

class UniversalEncoder(nn.Module):
    """
    Universal encoder supporting:
    - Input: raw coords or SH(L=10)
    - Activation: spline, relu, or siren
    """
    def __init__(self, input_type='sh', sh_legendre_polys=10,
                 activation_type='spline', activation_kwargs=None,
                 n_layers=3, hidden_dim=256, output_dim=256):
        super().__init__()
        self.input_type = input_type
        self.activation_type = activation_type
        
        # Input encoding
        if input_type == 'raw':
            self.posenc = None
            input_dim = 2
        elif input_type == 'sh':
            self.posenc = PE.SphericalHarmonics(
                legendre_polys=sh_legendre_polys,
                harmonics_calculation='analytic'
            )
            with torch.no_grad():
                test_coords = torch.zeros(1, 2)
                test_output = self.posenc(test_coords)
                input_dim = test_output.shape[1]
        else:
            raise ValueError(f"Unknown input_type: {input_type}")
        
        if activation_kwargs is None:
            activation_kwargs = {}
        
        # Build network
        dims = [input_dim] + [hidden_dim] * n_layers + [output_dim]
        
        if activation_type == 'siren':
            self.layers = nn.ModuleList()
            for i in range(len(dims) - 1):
                is_first = (i == 0)
                w0 = 30.0 if is_first else 1.0
                if i < len(dims) - 2:
                    self.layers.append(SirenLayer(dims[i], dims[i+1], w0=w0, is_first=is_first))
                else:
                    linear = nn.Linear(dims[i], dims[i+1])
                    bound = np.sqrt(6.0 / dims[i]) / 1.0
                    linear.weight.data.uniform_(-bound, bound)
                    if linear.bias is not None:
                        linear.bias.data.uniform_(-bound, bound)
                    self.layers.append(linear)
            self.activations = None
        
        elif activation_type == 'spline':
            self.linears = nn.ModuleList([
                nn.Linear(dims[i], dims[i+1])
                for i in range(len(dims) - 1)
            ])
            
            self.activations = nn.ModuleList([
                SplineActivation(**activation_kwargs)
                for _ in range(n_layers)
            ])
            
            for linear in self.linears:
                nn.init.kaiming_normal_(linear.weight)
                nn.init.zeros_(linear.bias)
        
        elif activation_type == 'relu':
            layers = []
            for i in range(len(dims) - 1):
                layers.append(nn.Linear(dims[i], dims[i+1]))
                if i < len(dims) - 2:
                    layers.append(nn.ReLU())
            self.net = nn.Sequential(*layers)
            for m in self.modules():
                if isinstance(m, nn.Linear):
                    nn.init.kaiming_normal_(m.weight)
                    nn.init.zeros_(m.bias)
        
        else:
            raise ValueError(f"Unknown activation_type: {activation_type}")
    
    def forward(self, coords):
        # Input encoding
        if self.input_type == 'raw':
            x = coords / torch.tensor([180., 90.], device=coords.device)
        else:  # 'sh'
            x = self.posenc(coords)
        
        # Forward through network
        if self.activation_type == 'siren':
            for layer in self.layers:
                x = layer(x)
        elif self.activation_type == 'spline':
            for i, (linear, act) in enumerate(zip(self.linears[:-1], self.activations)):
                x = act(linear(x))
            x = self.linears[-1](x)
        else:  # 'relu'
            x = self.net(x)
        
        return x


print("✅ Universal encoder loaded!")

---
## Training Utilities

In [ ]:
def sample_blocked(data, lons, lats, n_samples=15000, grid_size=5.0, test_ratio=0.3, seed=42):
    """
    Sample data with spatial blocking for train/test split.
    
    Args:
        data: 2D array of values
        lons: 1D array of longitude values
        lats: 1D array of latitude values
        n_samples: Number of samples to draw
        grid_size: Size of spatial blocks (degrees)
        test_ratio: Fraction of blocks for test set
        seed: Random seed
    
    Returns:
        coords_train, vals_train, coords_test, vals_test
    """
    np.random.seed(seed)
    valid = data > -1e30
    
    n_lon = int(360 / grid_size)
    n_lat = int(180 / grid_size)
    n_cells = n_lon * n_lat
    
    test_cells = set(np.random.choice(n_cells, int(n_cells * test_ratio), replace=False))
    
    valid_idx = np.where(valid)
    n_valid = len(valid_idx[0])
    sample_idx = np.random.choice(n_valid, min(n_samples, n_valid), replace=False)
    
    rows, cols = valid_idx[0][sample_idx], valid_idx[1][sample_idx]
    sample_lons, sample_lats = lons[cols], lats[rows]
    sample_vals = data[rows, cols]
    
    train_mask = []
    for lon, lat in zip(sample_lons, sample_lats):
        cell = int((lat + 90) / grid_size) * n_lon + int((lon + 180) / grid_size)
        cell = min(cell, n_cells - 1)
        train_mask.append(cell not in test_cells)
    train_mask = np.array(train_mask)
    
    coords = np.stack([sample_lons, sample_lats], axis=1)
    return coords[train_mask], sample_vals[train_mask], coords[~train_mask], sample_vals[~train_mask]


class RegressionPredictor(nn.Module):
    """Prediction head for regression tasks."""
    def __init__(self, encoder):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, 1)
        )
    
    def forward(self, coords):
        return self.head(self.encoder(coords)).squeeze(-1)


class ClassificationPredictor(nn.Module):
    """Prediction head for classification tasks."""
    def __init__(self, encoder, n_classes=100):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(
            nn.Linear(256, 128), nn.ReLU(), nn.Linear(128, n_classes)
        )
    
    def forward(self, coords):
        return self.head(self.encoder(coords))


def train_regression(name, encoder, coords_train, vals_train, coords_test, vals_test,
                    epochs=100, batch_size=256, lr=1e-3, verbose=False):
    """Train encoder on regression task."""
    model = RegressionPredictor(encoder).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    
    train_X = torch.tensor(coords_train, dtype=torch.float32)
    train_y = torch.tensor(np.log1p(vals_train), dtype=torch.float32)
    test_X = torch.tensor(coords_test, dtype=torch.float32).to(device)
    test_y = torch.tensor(np.log1p(vals_test), dtype=torch.float32)
    
    loader = DataLoader(TensorDataset(train_X, train_y),
                       batch_size=batch_size, shuffle=True)
    
    best_r2 = -float('inf')
    start = time.time()
    
    for epoch in range(epochs):
        model.train()
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            loss = loss_fn(model(X), y)
            loss.backward()
            opt.step()
        
        if (epoch + 1) % 20 == 0 or epoch == 0:
            model.eval()
            with torch.no_grad():
                pred = model(test_X).cpu().numpy()
            r2 = r2_score(test_y.numpy(), pred)
            best_r2 = max(best_r2, r2)
            if verbose:
                print(f"  Epoch {epoch+1}/{epochs}: R² = {r2:.4f}")
    
    train_time = time.time() - start
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    return {
        'model': name,
        'task': 'regression',
        'r2': best_r2,
        'params': n_params,
        'time': train_time,
        'trained_model': model
    }


def train_classification(name, encoder, coords_train, vals_train, coords_test, vals_test,
                        n_bins=100, epochs=100, batch_size=256, lr=1e-3, verbose=False):
    """Train encoder on classification task."""
    model = ClassificationPredictor(encoder, n_classes=n_bins).to(device)
    opt = optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.CrossEntropyLoss()
    
    # Bin continuous values
    log_vals = np.log1p(vals_train)
    bins = np.linspace(log_vals.min(), log_vals.max(), n_bins + 1)
    train_labels = np.digitize(log_vals, bins) - 1
    train_labels = np.clip(train_labels, 0, n_bins - 1)
    
    log_test = np.log1p(vals_test)
    test_labels = np.digitize(log_test, bins) - 1
    test_labels = np.clip(test_labels, 0, n_bins - 1)
    
    train_X = torch.tensor(coords_train, dtype=torch.float32)
    train_y = torch.tensor(train_labels, dtype=torch.long)
    test_X = torch.tensor(coords_test, dtype=torch.float32).to(device)
    test_y = torch.tensor(test_labels, dtype=torch.long)
    
    loader = DataLoader(TensorDataset(train_X, train_y),
                       batch_size=batch_size, shuffle=True)
    
    best_acc = 0.0
    start = time.time()
    
    for epoch in range(epochs):
        model.train()
        for X, y in loader:
            X, y = X.to(device), y.to(device)
            opt.zero_grad()
            loss = loss_fn(model(X), y)
            loss.backward()
            opt.step()
        
        if (epoch + 1) % 20 == 0 or epoch == 0:
            model.eval()
            with torch.no_grad():
                pred = model(test_X).cpu().numpy().argmax(axis=1)
            acc = (pred == test_y.numpy()).mean()
            best_acc = max(best_acc, acc)
            if verbose:
                print(f"  Epoch {epoch+1}/{epochs}: Acc = {acc:.4f}")
    
    train_time = time.time() - start
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    
    return {
        'model': name,
        'task': 'classification',
        'accuracy': best_acc,
        'params': n_params,
        'time': train_time,
        'trained_model': model
    }


print("✅ Training utilities loaded!")

---
## Experiment 1: Regression vs Classification ⭐⭐⭐ (CRITICAL)

**Hypothesis**: Learned activations (splines) help regression more than classification.

**Setup**: Test same data (population) with different loss functions.

**Expected**: Spline > ReLU for regression, ReLU ≈ Spline for classification.

In [ ]:
print("="*80)
print("EXPERIMENT 1: REGRESSION VS CLASSIFICATION")
print("="*80)

# Prepare population data
if pop_data is not None:
    coords_train_pop, vals_train_pop, coords_test_pop, vals_test_pop = sample_blocked(
        pop_data, lons_pop, lats_pop, n_samples=15000
    )
    print(f"\nPopulation data: {len(coords_train_pop)} train, {len(coords_test_pop)} test")
    
    results_exp1 = []
    activations = ['relu', 'spline', 'siren']
    
    # Test Regression
    print("\n" + "-"*80)
    print("REGRESSION FORMULATION")
    print("-"*80)
    
    for act in activations:
        print(f"\nTesting {act.upper()}...")
        
        if act == 'spline':
            kwargs = {'n_knots': 15, 'init': 'relu'}
        else:
            kwargs = None
        
        enc = UniversalEncoder(
            input_type='sh',
            sh_legendre_polys=10,
            activation_type=act,
            activation_kwargs=kwargs
        )
        
        res = train_regression(
            f'SH + {act.upper()} (reg)', enc,
            coords_train_pop, vals_train_pop,
            coords_test_pop, vals_test_pop,
            verbose=True
        )
        results_exp1.append(res)
        print(f"  Final R²: {res['r2']:.4f}")
    
    # Test Classification
    print("\n" + "-"*80)
    print("CLASSIFICATION FORMULATION")
    print("-"*80)
    
    for act in activations:
        print(f"\nTesting {act.upper()}...")
        
        if act == 'spline':
            kwargs = {'n_knots': 15, 'init': 'relu'}
        else:
            kwargs = None
        
        enc = UniversalEncoder(
            input_type='sh',
            sh_legendre_polys=10,
            activation_type=act,
            activation_kwargs=kwargs
        )
        
        res = train_classification(
            f'SH + {act.upper()} (cls)', enc,
            coords_train_pop, vals_train_pop,
            coords_test_pop, vals_test_pop,
            verbose=True
        )
        results_exp1.append(res)
        print(f"  Final Acc: {res['accuracy']:.4f}")
    
    # Summary
    df_exp1 = pd.DataFrame(results_exp1)
    print("\n" + "="*80)
    print("EXPERIMENT 1 SUMMARY")
    print("="*80)
    print(df_exp1[['model', 'task', 'r2', 'accuracy', 'time']].fillna('-').to_string(index=False))
    print("="*80)
    
    # Save results
    df_exp1.to_csv('exp1_regression_vs_classification.csv', index=False)
    print("\n✅ Results saved to exp1_regression_vs_classification.csv")
else:
    print("\n❌ Population data not available, skipping Experiment 1")

---
## Experiment 2: High-Frequency Geographic Tasks ⭐⭐⭐ (CRITICAL)

**Hypothesis**: Learned activations excel on high-frequency tasks (elevation, coastlines).

**Setup**: Test on 3 tasks of increasing frequency content:
1. Population (smooth) - baseline
2. Elevation (sharp peaks/valleys)
3. Coastline distance (step functions)

**Expected**: Spline advantage increases with frequency content.

In [ ]:
print("="*80)
print("EXPERIMENT 2: HIGH-FREQUENCY GEOGRAPHIC TASKS")
print("="*80)

results_exp2 = []
activations = ['relu', 'spline', 'siren']

# Task A: Population (baseline - smooth)
if pop_data is not None:
    print("\n" + "-"*80)
    print("TASK A: POPULATION DENSITY (SMOOTH BASELINE)")
    print("-"*80)
    
    for act in activations:
        print(f"\nTesting {act.upper()} on population...")
        
        if act == 'spline':
            kwargs = {'n_knots': 15, 'init': 'relu'}
        else:
            kwargs = None
        
        enc = UniversalEncoder(
            input_type='sh',
            sh_legendre_polys=10,
            activation_type=act,
            activation_kwargs=kwargs
        )
        
        res = train_regression(
            f'{act.upper()} - Population', enc,
            coords_train_pop, vals_train_pop,
            coords_test_pop, vals_test_pop,
            verbose=True
        )
        res['task_name'] = 'Population'
        res['frequency'] = 'low'
        results_exp2.append(res)
        print(f"  Final R²: {res['r2']:.4f}")

# Task B: Elevation (high-frequency)
if elevation_data is not None:
    print("\n" + "-"*80)
    print("TASK B: ELEVATION (HIGH-FREQUENCY)")
    print("-"*80)
    
    # Sample elevation data
    coords_train_elev, vals_train_elev, coords_test_elev, vals_test_elev = sample_blocked(
        elevation_data, lons_elev, lats_elev, n_samples=15000
    )
    print(f"Elevation data: {len(coords_train_elev)} train, {len(coords_test_elev)} test")
    
    # Normalize elevation (can be negative)
    elev_mean = vals_train_elev.mean()
    elev_std = vals_train_elev.std()
    vals_train_elev_norm = (vals_train_elev - elev_mean) / elev_std
    vals_test_elev_norm = (vals_test_elev - elev_mean) / elev_std
    
    # Shift to positive for log1p
    shift = abs(vals_train_elev_norm.min()) + 1
    vals_train_elev_pos = vals_train_elev_norm + shift
    vals_test_elev_pos = vals_test_elev_norm + shift
    
    for act in activations:
        print(f"\nTesting {act.upper()} on elevation...")
        
        if act == 'spline':
            kwargs = {'n_knots': 15, 'init': 'relu'}
        else:
            kwargs = None
        
        enc = UniversalEncoder(
            input_type='sh',
            sh_legendre_polys=10,
            activation_type=act,
            activation_kwargs=kwargs
        )
        
        res = train_regression(
            f'{act.upper()} - Elevation', enc,
            coords_train_elev, vals_train_elev_pos,
            coords_test_elev, vals_test_elev_pos,
            verbose=True
        )
        res['task_name'] = 'Elevation'
        res['frequency'] = 'high'
        results_exp2.append(res)
        print(f"  Final R²: {res['r2']:.4f}")
else:
    print("\n⚠️  Elevation data not available, skipping Task B")

# Summary
if len(results_exp2) > 0:
    df_exp2 = pd.DataFrame(results_exp2)
    print("\n" + "="*80)
    print("EXPERIMENT 2 SUMMARY")
    print("="*80)
    print(df_exp2[['model', 'task_name', 'frequency', 'r2', 'time']].to_string(index=False))
    print("="*80)
    
    # Calculate spline advantage
    print("\nSPLINE ADVANTAGE BY TASK:")
    for task in df_exp2['task_name'].unique():
        task_df = df_exp2[df_exp2['task_name'] == task]
        relu_r2 = task_df[task_df['model'].str.contains('RELU')]['r2'].values[0]
        spline_r2 = task_df[task_df['model'].str.contains('SPLINE')]['r2'].values[0]
        advantage = spline_r2 - relu_r2
        print(f"  {task:20s}: {advantage:+.4f} ({100*advantage/relu_r2:+.2f}%)")
    
    # Save results
    df_exp2.to_csv('exp2_high_frequency_tasks.csv', index=False)
    print("\n✅ Results saved to exp2_high_frequency_tasks.csv")
else:
    print("\n❌ No data available for Experiment 2")

---
## Experiment 3: Multi-Resolution Analysis ⭐⭐ (HIGH PRIORITY)

**Hypothesis**: Finer resolution → more high-frequency detail → spline advantage.

**Setup**: Downsample elevation data to multiple resolutions and test.

**Expected**: Spline advantage increases with resolution.

In [ ]:
print("="*80)
print("EXPERIMENT 3: MULTI-RESOLUTION ANALYSIS")
print("="*80)

if elevation_data is not None:
    from scipy.ndimage import zoom
    
    results_exp3 = []
    activations = ['relu', 'spline']
    
    # Define resolutions (downsampling factors)
    resolutions = [
        {'name': 'coarse',  'factor': 0.25, 'n_samples': 5000},   # 4x downsample
        {'name': 'medium',  'factor': 0.5,  'n_samples': 10000},  # 2x downsample
        {'name': 'fine',    'factor': 1.0,  'n_samples': 15000},  # Full resolution
    ]
    
    for res_config in resolutions:
        name = res_config['name']
        factor = res_config['factor']
        n_samples = res_config['n_samples']
        
        print(f"\n" + "-"*80)
        print(f"RESOLUTION: {name.upper()} (factor={factor})")
        print("-"*80)
        
        # Downsample if needed
        if factor < 1.0:
            elev_resampled = zoom(elevation_data, factor, order=1)
            lons_resampled = np.linspace(-180, 180, elev_resampled.shape[1])
            lats_resampled = np.linspace(90, -90, elev_resampled.shape[0])
        else:
            elev_resampled = elevation_data
            lons_resampled = lons_elev
            lats_resampled = lats_elev
        
        print(f"Resampled shape: {elev_resampled.shape}")
        
        # Sample data
        coords_train, vals_train, coords_test, vals_test = sample_blocked(
            elev_resampled, lons_resampled, lats_resampled, n_samples=n_samples
        )
        
        # Normalize
        elev_mean = vals_train.mean()
        elev_std = vals_train.std()
        vals_train_norm = (vals_train - elev_mean) / elev_std
        vals_test_norm = (vals_test - elev_mean) / elev_std
        shift = abs(vals_train_norm.min()) + 1
        vals_train_pos = vals_train_norm + shift
        vals_test_pos = vals_test_norm + shift
        
        print(f"Sampled: {len(coords_train)} train, {len(coords_test)} test")
        
        for act in activations:
            print(f"\nTesting {act.upper()} at {name} resolution...")
            
            if act == 'spline':
                kwargs = {'n_knots': 15, 'init': 'relu'}
            else:
                kwargs = None
            
            enc = UniversalEncoder(
                input_type='sh',
                sh_legendre_polys=10,
                activation_type=act,
                activation_kwargs=kwargs
            )
            
            res = train_regression(
                f'{act.upper()} - {name}', enc,
                coords_train, vals_train_pos,
                coords_test, vals_test_pos,
                verbose=True,
                epochs=80  # Slightly shorter for efficiency
            )
            res['resolution'] = name
            res['downsample_factor'] = factor
            results_exp3.append(res)
            print(f"  Final R²: {res['r2']:.4f}")
    
    # Summary
    df_exp3 = pd.DataFrame(results_exp3)
    print("\n" + "="*80)
    print("EXPERIMENT 3 SUMMARY")
    print("="*80)
    print(df_exp3[['model', 'resolution', 'r2', 'time']].to_string(index=False))
    print("="*80)
    
    # Calculate spline advantage by resolution
    print("\nSPLINE ADVANTAGE BY RESOLUTION:")
    for res in ['coarse', 'medium', 'fine']:
        res_df = df_exp3[df_exp3['resolution'] == res]
        if len(res_df) == 2:
            relu_r2 = res_df[res_df['model'].str.contains('RELU')]['r2'].values[0]
            spline_r2 = res_df[res_df['model'].str.contains('SPLINE')]['r2'].values[0]
            advantage = spline_r2 - relu_r2
            print(f"  {res:10s}: {advantage:+.4f} ({100*advantage/relu_r2:+.2f}%)")
    
    # Save results
    df_exp3.to_csv('exp3_multi_resolution.csv', index=False)
    print("\n✅ Results saved to exp3_multi_resolution.csv")
    
else:
    print("\n❌ Elevation data not available, skipping Experiment 3")

---
## Experiment 4: Function Complexity Measurement ⭐⭐

**Method**: Measure Total Variation (TV) along random paths in input space.

**Hypothesis**: Complexity correlates with performance for regression (not classification).

**Expected**: Spline models have higher TV than ReLU models.

In [ ]:
print("="*80)
print("EXPERIMENT 4: FUNCTION COMPLEXITY MEASUREMENT")
print("="*80)

def compute_total_variation(model, test_coords, n_paths=500, n_steps=100):
    """
    Compute Total Variation (TV) along random paths in input space.
    
    Args:
        model: Trained neural network
        test_coords: Test coordinates for sampling endpoints
        n_paths: Number of random paths to average over
        n_steps: Number of steps along each path
    
    Returns:
        tv: Total variation (complexity measure)
    """
    model.eval()
    tv_values = []
    
    test_coords_tensor = torch.tensor(test_coords, dtype=torch.float32).to(device)
    
    for _ in range(n_paths):
        # Sample two random points
        idx = np.random.choice(len(test_coords), 2, replace=False)
        x1 = test_coords_tensor[idx[0]]
        x2 = test_coords_tensor[idx[1]]
        
        # Create path: x(λ) = (1-λ)x1 + λx2, λ ∈ [0,1]
        lambdas = torch.linspace(0, 1, n_steps).to(device)
        path = torch.stack([(1-lam)*x1 + lam*x2 for lam in lambdas])
        
        # Evaluate model along path
        with torch.no_grad():
            outputs = model(path).cpu().numpy()
        
        # Compute TV: sum of |f(xi+1) - f(xi)|
        tv = np.sum(np.abs(np.diff(outputs)))
        tv_values.append(tv)
    
    return np.mean(tv_values)


# Measure complexity for models from Experiment 1 and 2
if len(results_exp1) > 0:
    print("\nMeasuring complexity for models...")
    
    complexity_results = []
    
    # Get regression models from Exp 1
    for res in results_exp1:
        if res['task'] == 'regression':
            print(f"\nComputing TV for {res['model']}...")
            model = res['trained_model']
            tv = compute_total_variation(model, coords_test_pop, n_paths=500)
            complexity_results.append({
                'model': res['model'],
                'task': 'population',
                'formulation': 'regression',
                'r2': res['r2'],
                'total_variation': tv
            })
            print(f"  TV = {tv:.2f}")
    
    # Summary
    df_exp4 = pd.DataFrame(complexity_results)
    print("\n" + "="*80)
    print("EXPERIMENT 4 SUMMARY")
    print("="*80)
    print(df_exp4[['model', 'r2', 'total_variation']].to_string(index=False))
    print("="*80)
    
    # Check correlation
    if len(df_exp4) >= 3:
        from scipy.stats import pearsonr
        corr, pval = pearsonr(df_exp4['r2'], df_exp4['total_variation'])
        print(f"\nCorrelation (R² vs TV): r = {corr:.3f}, p = {pval:.3f}")
        if pval < 0.05:
            print("✅ Significant positive correlation!")
        else:
            print("⚠️  Correlation not significant")
    
    # Save results
    df_exp4.to_csv('exp4_complexity_measurement.csv', index=False)
    print("\n✅ Results saved to exp4_complexity_measurement.csv")
else:
    print("\n❌ No trained models available for complexity measurement")

---
## Experiment 5: Task Difficulty Scaling ⭐

**Hypothesis**: Harder tasks → more benefit from learned activations.

**Setup**: Create tasks of varying difficulty by smoothing elevation data.

**Expected**: Spline advantage increases with task difficulty.

In [ ]:
print("="*80)
print("EXPERIMENT 5: TASK DIFFICULTY SCALING")
print("="*80)

if elevation_data is not None:
    from scipy.ndimage import gaussian_filter
    
    results_exp5 = []
    activations = ['relu', 'spline']
    
    # Define difficulty levels (smoothing sigma)
    difficulties = [
        {'name': 'easy',   'sigma': 10.0},  # Heavy smoothing
        {'name': 'medium', 'sigma': 3.0},   # Moderate smoothing
        {'name': 'hard',   'sigma': 0.0},   # No smoothing (raw)
    ]
    
    for diff_config in difficulties:
        name = diff_config['name']
        sigma = diff_config['sigma']
        
        print(f"\n" + "-"*80)
        print(f"DIFFICULTY: {name.upper()} (sigma={sigma})")
        print("-"*80)
        
        # Apply smoothing
        if sigma > 0:
            elev_smoothed = gaussian_filter(elevation_data.astype(float), sigma=sigma)
        else:
            elev_smoothed = elevation_data
        
        # Sample data
        coords_train, vals_train, coords_test, vals_test = sample_blocked(
            elev_smoothed, lons_elev, lats_elev, n_samples=10000
        )
        
        # Normalize
        elev_mean = vals_train.mean()
        elev_std = vals_train.std()
        vals_train_norm = (vals_train - elev_mean) / elev_std
        vals_test_norm = (vals_test - elev_mean) / elev_std
        shift = abs(vals_train_norm.min()) + 1
        vals_train_pos = vals_train_norm + shift
        vals_test_pos = vals_test_norm + shift
        
        print(f"Sampled: {len(coords_train)} train, {len(coords_test)} test")
        
        for act in activations:
            print(f"\nTesting {act.upper()} on {name} task...")
            
            if act == 'spline':
                kwargs = {'n_knots': 15, 'init': 'relu'}
            else:
                kwargs = None
            
            enc = UniversalEncoder(
                input_type='sh',
                sh_legendre_polys=10,
                activation_type=act,
                activation_kwargs=kwargs
            )
            
            res = train_regression(
                f'{act.upper()} - {name}', enc,
                coords_train, vals_train_pos,
                coords_test, vals_test_pos,
                verbose=True,
                epochs=80
            )
            res['difficulty'] = name
            res['smoothing_sigma'] = sigma
            results_exp5.append(res)
            print(f"  Final R²: {res['r2']:.4f}")
    
    # Summary
    df_exp5 = pd.DataFrame(results_exp5)
    print("\n" + "="*80)
    print("EXPERIMENT 5 SUMMARY")
    print("="*80)
    print(df_exp5[['model', 'difficulty', 'r2', 'time']].to_string(index=False))
    print("="*80)
    
    # Calculate spline advantage by difficulty
    print("\nSPLINE ADVANTAGE BY DIFFICULTY:")
    for diff in ['easy', 'medium', 'hard']:
        diff_df = df_exp5[df_exp5['difficulty'] == diff]
        if len(diff_df) == 2:
            relu_r2 = diff_df[diff_df['model'].str.contains('RELU')]['r2'].values[0]
            spline_r2 = diff_df[diff_df['model'].str.contains('SPLINE')]['r2'].values[0]
            advantage = spline_r2 - relu_r2
            print(f"  {diff:10s}: {advantage:+.4f} ({100*advantage/relu_r2:+.2f}%)")
    
    # Save results
    df_exp5.to_csv('exp5_task_difficulty.csv', index=False)
    print("\n✅ Results saved to exp5_task_difficulty.csv")
    
else:
    print("\n❌ Elevation data not available, skipping Experiment 5")

---
## Final Summary

Synthesize results across all experiments.

In [ ]:
print("="*80)
print("NOTEBOOK 19: FINAL SUMMARY")
print("="*80)

print("\n📊 KEY FINDINGS:\n")

# Experiment 1
if len(results_exp1) > 0:
    print("1. REGRESSION VS CLASSIFICATION:")
    df1_reg = pd.DataFrame([r for r in results_exp1 if r['task'] == 'regression'])
    df1_cls = pd.DataFrame([r for r in results_exp1 if r['task'] == 'classification'])
    
    if len(df1_reg) >= 2:
        relu_reg = df1_reg[df1_reg['model'].str.contains('RELU')]['r2'].values[0]
        spline_reg = df1_reg[df1_reg['model'].str.contains('SPLINE')]['r2'].values[0]
        reg_adv = spline_reg - relu_reg
        print(f"   - Regression: Spline vs ReLU = {reg_adv:+.4f} ({100*reg_adv/relu_reg:+.2f}%)")
    
    if len(df1_cls) >= 2:
        relu_cls = df1_cls[df1_cls['model'].str.contains('RELU')]['accuracy'].values[0]
        spline_cls = df1_cls[df1_cls['model'].str.contains('SPLINE')]['accuracy'].values[0]
        cls_adv = spline_cls - relu_cls
        print(f"   - Classification: Spline vs ReLU = {cls_adv:+.4f} ({100*cls_adv/relu_cls:+.2f}%)")

# Experiment 2
if len(results_exp2) > 0:
    print("\n2. HIGH-FREQUENCY TASKS:")
    df2 = pd.DataFrame(results_exp2)
    for task in df2['task_name'].unique():
        task_df = df2[df2['task_name'] == task]
        if len(task_df) >= 2:
            relu_r2 = task_df[task_df['model'].str.contains('RELU')]['r2'].values[0]
            spline_r2 = task_df[task_df['model'].str.contains('SPLINE')]['r2'].values[0]
            advantage = spline_r2 - relu_r2
            print(f"   - {task}: Spline vs ReLU = {advantage:+.4f} ({100*advantage/relu_r2:+.2f}%)")

# Experiment 3
if len(results_exp3) > 0:
    print("\n3. MULTI-RESOLUTION:")
    df3 = pd.DataFrame(results_exp3)
    for res in df3['resolution'].unique():
        res_df = df3[df3['resolution'] == res]
        if len(res_df) >= 2:
            relu_r2 = res_df[res_df['model'].str.contains('RELU')]['r2'].values[0]
            spline_r2 = res_df[res_df['model'].str.contains('SPLINE')]['r2'].values[0]
            advantage = spline_r2 - relu_r2
            print(f"   - {res} resolution: Spline vs ReLU = {advantage:+.4f} ({100*advantage/relu_r2:+.2f}%)")

# Experiment 4
if 'df_exp4' in locals() and len(df_exp4) > 0:
    print("\n4. FUNCTION COMPLEXITY:")
    for _, row in df_exp4.iterrows():
        print(f"   - {row['model']:30s}: TV = {row['total_variation']:.2f}, R² = {row['r2']:.4f}")

# Experiment 5
if len(results_exp5) > 0:
    print("\n5. TASK DIFFICULTY:")
    df5 = pd.DataFrame(results_exp5)
    for diff in df5['difficulty'].unique():
        diff_df = df5[df5['difficulty'] == diff]
        if len(diff_df) >= 2:
            relu_r2 = diff_df[diff_df['model'].str.contains('RELU')]['r2'].values[0]
            spline_r2 = diff_df[diff_df['model'].str.contains('SPLINE')]['r2'].values[0]
            advantage = spline_r2 - relu_r2
            print(f"   - {diff} difficulty: Spline vs ReLU = {advantage:+.4f} ({100*advantage/relu_r2:+.2f}%)")

print("\n" + "="*80)
print("🎯 CONCLUSION: Did we find \"alpha\"?")
print("="*80)
print("\nCheck the results above to determine:")
print("1. Does Spline beat ReLU on regression? (Exp 1)")
print("2. Does Spline beat ReLU on elevation? (Exp 2)")
print("3. Does Spline advantage increase with resolution? (Exp 3)")
print("4. Does complexity correlate with performance? (Exp 4)")
print("5. Does Spline advantage increase with difficulty? (Exp 5)")
print("\nIf YES to ANY of these → We found alpha!")
print("="*80)

print("\n✅ Notebook 19 complete!")
print("\nNext steps:")
print("- Review all CSV files for detailed results")
print("- Create ANALYSIS_NOTEBOOK19.md summarizing findings")
print("- Update README.md with key conclusions")
print("- Proceed to Notebook 20 (visualization) if alpha found")

---
## Conclusions

### What We Tested:

1. **Regression vs Classification**: Does formulation matter?
2. **High-Frequency Tasks**: Does spline excel on sharp transitions?
3. **Multi-Resolution**: Does finer resolution help splines?
4. **Function Complexity**: Does TV correlate with performance?
5. **Task Difficulty**: Does spline advantage scale with difficulty?

### Expected Outcomes:

Based on **Teney et al. (2024)**:
- ✅ Spline > ReLU on regression (not classification)
- ✅ Spline > ReLU on elevation (high-frequency)
- ✅ Spline advantage increases with resolution
- ✅ Complexity (TV) correlates with performance
- ✅ Spline advantage increases with task difficulty

### Next Steps:
- Analyze results and create **ANALYSIS_NOTEBOOK19.md**
- Visualize learned activation shapes (Notebook 20)
- Test robustness with multiple seeds (Notebook 21)